# Stage 2: does computation return to the pretrained recurrent backbone?

**Run with Runtime → Run all. Nothing to set.** Reconnect and run all again after a disconnect;
finished jobs load from Drive. Send back `STAGE2_SUMMARY.txt`.

## What this decides

Stage 1 produced a pretrained Mamba backbone with an added attention route that took over a task
the backbone could already do alone. That is the pretrained version of the paper's *free retrieval
is the default* result. This notebook asks the question the paper needs answered:

> After convergence, can pricing the attention route hand the computation back to the pretrained
> recurrent backbone, at the conditions the frozen-backbone screen called viable and not at the
> conditions it called non-viable?

## Three things that differ from Stage 1, deliberately

**The backbone is unfrozen during steering.** In Stage 1 it was frozen, which is why the added
route learned a nearly prompt-independent enabling bias and resampling read zero. Nothing could
receive the computation. Here every parameter trains. That creates the obvious objection — maybe
the backbone simply relearns the task — which is why the `never` arm exists and why the headline
comparison is *paired against it*, not against the pre-switch checkpoint.

**Deletion is the primary readout.** Appendix F of the paper is the reason: resampling measures
trial-varying information and can read zero on a route that is functionally necessary. So the
headline number is accuracy with every attention branch zeroed. Resampling `D_attn` / `D_ssm` is
still logged for continuity with the rest of the paper, and the common-mode decomposition
(across-prompt mean vs per-prompt deviation) is logged at every evaluation so that if the route
is again mostly an enabling bias you see it immediately rather than after a full steering run.

**The price is selected before the arms are run.** There is no separable logit here — attention
writes into the residual stream — so the price is on the attention write magnitude, normalised by
its value at the Stage 1 checkpoint so that λ is comparable across models. A short dose–response
sweep picks λ and writes it to `price.json` *before* any arm runs, mirroring Figure 5.

## Honest caveat to carry into the paper

Pricing a stream-write magnitude is exactly the intervention that did **not** move causal
dependence in fused MQAR, because those branches were composed rather than substitutable. If the
same thing happens here, that is an informative negative for the prediction table, not a bug: the
viability screen passed and handover still failed, which would sharpen the claim to *viability is
necessary but not sufficient*. Both outcomes are reportable. Do not tune your way out of it.

## Kaggle notes

- **Internet must be ON** (Settings → Internet) or the backbone and WikiText will not download.
- Stage 1 lives elsewhere. Upload its folder (the one containing `checkpoints_*/` and
  `results_*/boundary.json`) as a **Kaggle Dataset** and attach it; this notebook searches
  `/kaggle/input` recursively for `phaseA_final_*.pt` and `boundary.json`, so no path to set.
- Everything is written to `/kaggle/working`, which only survives if you **Save Version →
  Save & Run All**. To continue a partial run in a new session, attach *this* notebook's output
  as a dataset alongside the Stage 1 one; finished jobs are then found in `records.jsonl` and
  skipped, and a paused job resumes from its checkpoint.
- P100 and T4 have no bfloat16, so the notebook runs in **float32 by default**. Mamba's
  sequential scan can overflow in float16, and a silent overflow here would look like a failed
  handover. If it is too slow, set `FORCE_FP16 = True`; a GradScaler is already wired in.

## Order of work

Jobs run in priority order and each one is checkpointed, so a partial run is still usable:

1. viable condition — `late` and `never`, seed 0 (the matched pair that carries the claim)
2. the same pair at seeds 1 and 2
3. non-viable condition — `late` (the boundary row: should *fail* to hand over)
4. viable condition — `withdrawn` (behavioural reversibility)

In [ ]:
import glob, hashlib, json, math, os, pathlib, random, subprocess, sys, time
from contextlib import contextmanager, nullcontext

PRESET = "full"           # "smoke" = tiny budgets, for checking the notebook runs end to end
FORCE_FP16 = False        # see the Kaggle note: float32 by default on P100/T4
STAGE1_MODEL = None       # set e.g. "AntonV/mamba2-130m-hf" only if auto-discovery fails

IN_KAGGLE = pathlib.Path("/kaggle/working").exists()
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_KAGGLE:
    WORKDIR = "/kaggle/working/mamba_attn_route_s2"
    SEARCH_ROOTS = [pathlib.Path("/kaggle/input"), pathlib.Path("/kaggle/working")]
elif IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = "/content/drive/MyDrive/mamba_attn_route"
    SEARCH_ROOTS = [pathlib.Path(WORKDIR)]
else:
    WORKDIR = os.environ.get("MAMBA_WORKDIR", str(pathlib.Path.cwd() / "mamba_attn_route"))
    SEARCH_ROOTS = [pathlib.Path(WORKDIR)]

from importlib.metadata import version, PackageNotFoundError


def _ver(pkg):
    try:
        return tuple(int("".join(c for c in p if c.isdigit()) or 0) for p in version(pkg).split(".")[:2])
    except PackageNotFoundError:
        return (0, 0)


if _ver("transformers") < (4, 43):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "accelerate"], check=True)
    raise SystemExit("transformers upgraded. Runtime -> Restart session, then Run all again.")
if _ver("datasets") == (0, 0):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)

import importlib.util as _ilu
HAVE_KERNELS = all(_ilu.find_spec(m) is not None for m in ("mamba_ssm", "causal_conv1d"))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM


def P(full, smoke):
    return full if PRESET == "full" else smoke


# --- identical to Stage 1; the prompt format must not change or the screen no longer applies ---
N_ATTN_LAYERS, ATTN_HEADS = 8, 8
SCREEN_DIST, N_CAND = 64, 16
SEED, EVAL_SEED, FORMAT_VERSION = 0, 999, 1

# --- Stage 2 settings ---
STEER_STEPS   = P(400, 12)     # steps per arm after the switch
WITHDRAW_AT   = P(200, 6)      # in the withdrawn arm, price is removed at this step
EVAL_EVERY    = P(100, 6)
N_EVAL        = P(64, 8)       # prompts per evaluation (bigger than Stage 1: this is the headline)
TRAIN_BATCH   = P(8, 2)
EVAL_BATCH    = 8 if HAVE_KERNELS else 4
LR, WARMUP, GRAD_CLIP = 1e-5, 20, 1.0      # low LR: the whole pretrained model is training now
LM_W, LM_BATCH, LM_LEN = 1.0, 2, 256       # text replay, weighted up because the backbone is unfrozen
LAMBDAS       = P([0.01, 0.03, 0.1, 0.3, 1.0], [0.1])   # dose-response grid, normalised units
DOSE_STEPS    = P(120, 6)
SEEDS         = P([0, 1, 2], [0])
TIME_BUDGET_H = P(3.5, 0.2)
CKPT_MINUTES  = P(25.0, 2.0)
# price selection rule, fixed here before any number is seen:
SHARE_TARGET  = 0.30    # accept the smallest lambda whose attention write share falls below this
ACC_FLOOR     = 0.90    # ... while keeping at least this fraction of the pre-switch clean accuracy
# verdict rule, also fixed here:
HANDOVER_DEL  = 0.90    # deletion accuracy must reach this fraction of clean accuracy
HANDOVER_ACC  = 0.90    # clean accuracy must stay at this fraction of its pre-switch value

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IS_CUDA = DEVICE.type == "cuda"
GPU_NAME = torch.cuda.get_device_name(0) if IS_CUDA else "cpu"
CAP = torch.cuda.get_device_capability()[0] if IS_CUDA else 0
USE_BF16 = IS_CUDA and CAP >= 8
USE_FP16 = IS_CUDA and not USE_BF16 and FORCE_FP16
AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
USE_AMP = USE_BF16 or USE_FP16


def amp():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE) if USE_AMP else nullcontext()


try:
    SCALER = torch.amp.GradScaler("cuda", enabled=USE_FP16)
except (AttributeError, TypeError):
    SCALER = torch.cuda.amp.GradScaler(enabled=USE_FP16)


def sync():
    if IS_CUDA:
        torch.cuda.synchronize()


WORK = pathlib.Path(WORKDIR).expanduser()
PREC = "bf16 autocast" if USE_BF16 else ("fp16 autocast + GradScaler" if USE_FP16 else "float32")
print(f"{GPU_NAME} | {PREC} | torch {torch.__version__} | transformers {transformers.__version__} | "
      f"fast Mamba kernels: {HAVE_KERNELS} | kaggle: {IN_KAGGLE}")

In [ ]:
# Find the Stage 1 output: the phase A checkpoint, and the boundary that was recorded before it.
def find(pattern):
    hits = []
    for root in SEARCH_ROOTS:
        if root.exists():
            hits += sorted(glob.glob(str(root / "**" / pattern), recursive=True))
    return hits


cands = find("phaseA_final_*.pt")
assert cands, ("no Stage 1 phase-A checkpoint found. On Kaggle: upload the Stage 1 folder (the one "
               "with checkpoints_*/ and results_*/boundary.json) as a Dataset and attach it, then "
               "run all again. Searched: " + ", ".join(str(r) for r in SEARCH_ROOTS))
PHASE_A_CKPT = pathlib.Path(cands[-1])
TAG = PHASE_A_CKPT.parent.name[len("checkpoints_"):] if PHASE_A_CKPT.parent.name.startswith("checkpoints_") \
    else PHASE_A_CKPT.parent.name
bnds = find("boundary.json")
assert bnds, "found the phase-A checkpoint but no boundary.json; attach the Stage 1 results_* folder too"
BOUNDARY = json.loads(pathlib.Path(([b for b in bnds if TAG in b] or bnds)[-1]).read_text())
print(f"Stage 1 checkpoint: {PHASE_A_CKPT.name}\nboundary recorded {BOUNDARY['created']}: "
      f"viable {BOUNDARY['viable']}, not viable {BOUNDARY['nonviable']}")

MODEL_NAME = BOUNDARY.get("model") or STAGE1_MODEL
if MODEL_NAME is None:
    stem = TAG.rsplit("_", 1)[0]
    MODEL_NAME = next(m for m in (f"AntonV/{stem}", f"state-spaces/{stem}"))
print(f"backbone: {MODEL_NAME}")

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(DEVICE).eval()
model.config.use_cache = False

RESDIR, CKPTDIR, FIGDIR, TABDIR = (WORK / f"{n}_{TAG}_s2" for n in
                                   ("results", "checkpoints", "figures", "tables"))
for _d in (RESDIR, CKPTDIR, FIGDIR, TABDIR):
    _d.mkdir(parents=True, exist_ok=True)


def keyof(spec):
    return hashlib.md5(json.dumps(spec, sort_keys=True, default=str).encode()).hexdigest()[:16]


class Records:
    def __init__(self, path):
        self.path, self.recs = path, {}
        if path.exists():
            for line in path.read_text().splitlines():
                try:
                    r = json.loads(line); self.recs[r["_key"]] = r
                except (json.JSONDecodeError, KeyError):
                    pass

    def get(self, spec):
        return self.recs.get(keyof(spec))

    def put(self, spec, payload):
        r = {"_key": keyof(spec), "_spec": spec, "_t": time.time(), **payload}
        self.recs[r["_key"]] = r
        with open(self.path, "a") as f:
            f.write(json.dumps(r, default=float) + "\n"); f.flush(); os.fsync(f.fileno())
        return r


def ckpt_save(name, obj):
    p = CKPTDIR / f"{name}.pt"
    torch.save(obj, p.with_suffix(".tmp")); os.replace(p.with_suffix(".tmp"), p)


def ckpt_load(name):
    p = CKPTDIR / f"{name}.pt"
    if p.exists():
        try:
            return torch.load(p, map_location="cpu", weights_only=False)
        except Exception as e:
            print(f"  checkpoint unreadable ({type(e).__name__})")
    return None


def write_once(path, obj):
    if not path.exists():
        path.write_text(json.dumps(obj, indent=1))
    return json.loads(path.read_text())


REC = Records(RESDIR / "records.jsonl")
BASE_SPEC = {"model": MODEL_NAME, "stage": 2, "format": FORMAT_VERSION}


def find_layers(m):
    n = m.config.num_hidden_layers
    hits = [(name, mod) for name, mod in m.named_modules() if isinstance(mod, nn.ModuleList) and len(mod) == n]
    assert hits, f"no ModuleList of length {n}"
    return hits[0]


LAYER_PATH, LAYERS = find_layers(model)
N_LAYERS, D_MODEL = len(LAYERS), model.config.hidden_size
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters | {N_LAYERS} layers | "
      f"hidden {D_MODEL}")

In [ ]:
# Prompt construction, byte-identical to Stage 1 so the recorded boundary still applies.
NOUNS = """apple river table window garden doctor pencil island forest bottle mirror candle bridge castle rocket
tiger lemon piano jacket helmet market pocket village teacher engine planet carpet button ticket basket hammer
ladder wallet bucket camera coffee cookie dragon farmer finger flower guitar hunter kitten letter monkey needle
orange parrot pillow puzzle rabbit saddle sister spider statue sugar thunder tomato turtle wagon wizard anchor
arrow badge banner barrel beach beard bell belt bench blanket boat bone book boot bread brick brush cabin cable
cake canal cannon card cart chain chair cheese cherry chest chicken church cloud coat coin comet cotton crown cup
curtain desk diamond dinner dog door drum eagle egg elbow fence field flag flute fork fox frog glass glove goat
grape hat honey horse house iron jar jewel kettle key kite knife lake lamp leaf lion lock map mask milk moon
mouse nail nest net oven owl paint palace paper pearl pepper pig pipe plate pond pot queen rain ring road robot
roof rope rose salt scarf ship shirt shoe shovel sink skirt snake soap sock spoon star stone storm sword tent
tower train tree truck tunnel violin wall whale wheel wolf yard""".split()
ADJS = """red blue green yellow purple brown black white pink gray golden silent happy angry brave calm clever
cold dark eager early empty fair fancy fast fierce gentle giant grand great heavy hollow humble icy jolly kind
large late lazy little lonely loud lucky mild modern narrow neat noble odd old plain polite proud quick quiet
rapid rare rich rough round royal rusty sacred safe sharp shiny short shy simple sleepy slow small smooth soft
solid sour spicy steep sticky stormy strange strict strong sunny sweet swift tall tame thick thin tidy tiny tough
ugly vast warm weak wet wide wild wise young bitter bold bright broad busy cheap crisp cruel cute damp deep dry
dusty faint false fresh frozen full funny grumpy hairy harsh hot huge juicy keen lean lively loyal mad merry
messy misty muddy nasty naughty nervous nice pale pure rigid ripe rude salty scary shallow silly skinny spotty
stale stiff sturdy tender tense tricky vivid weird wooden zany""".split()


def single_token(words, exclude=()):
    ids = []
    for w in words:
        e = tok.encode(" " + w, add_special_tokens=False)
        if len(e) == 1 and e[0] not in ids and e[0] not in exclude:
            ids.append(e[0])
    return ids


KEY_IDS = single_token(NOUNS)
VAL_IDS = single_token(ADJS, exclude=set(KEY_IDS))


def piece(s):
    return tok.encode(s, add_special_tokens=False)


BOS = [tok.bos_token_id] if tok.bos_token_id is not None else []
_WIKI = None


def wiki_tokens():
    global _WIKI
    if _WIKI is None:
        from datasets import load_dataset
        try:
            ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="validation")
        except Exception:
            ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="validation")
        _WIKI = piece("\n\n".join(t for t in ds["text"] if t.strip()))
    return _WIKI


def make_prompts(load, dist, n, seed):
    W = wiki_tokens()
    rng = np.random.default_rng([seed, load, dist, FORMAT_VERSION])
    X, Y, C = [], [], []
    for _ in range(n):
        ks = rng.choice(len(KEY_IDS), load, replace=False)
        cand = rng.choice(len(VAL_IDS), N_CAND, replace=False)
        vs = cand[:load]
        q = int(rng.integers(load))
        ids = BOS + piece("Note these facts.\n")
        for k, v in zip(ks, vs):
            ids += piece("The") + [KEY_IDS[k]] + piece(" is") + [VAL_IDS[v]] + piece(".\n")
        s = int(rng.integers(len(W) - dist - 1))
        ids += piece("\n") + W[s:s + dist] + piece("\nQuestion: the") + [KEY_IDS[ks[q]]] + piece(" is")
        X.append(ids); Y.append(VAL_IDS[vs[q]]); C.append([VAL_IDS[c] for c in cand])
    L = max(len(x) for x in X)
    keep = [i for i, x in enumerate(X) if len(x) == L]
    return (torch.tensor([X[i] for i in keep]), torch.tensor([Y[i] for i in keep]),
            torch.tensor([C[i] for i in keep]))


print(f"single-token pools: {len(KEY_IDS)} keys, {len(VAL_IDS)} values | chance {1 / N_CAND:.3f}")

In [ ]:
# Route machinery. Same wrapper as Stage 1, plus three kinds Stage 2 needs:
#   "price"  accumulates the attention write magnitude WITH gradient, for the penalty
#   "cmean"  replaces the route by its across-prompt mean  (the enabling-bias component)
#   "dev"    keeps only the per-prompt deviation           (the trial-specific component)
ACTIVE = {}


def apply_iv(route, idx, t):
    spec = ACTIVE.get(route)
    if spec is None or t is None:
        return t
    kind = spec["kind"]
    if kind == "record":
        spec["store"].setdefault(idx, []).append(t.detach().clone())
        return t
    if kind == "price":
        spec["acc"].append(t.float().pow(2).sum(-1).mean())
        return t
    if kind == "zero":
        return torch.zeros_like(t)
    if kind == "cache":
        return spec["store"][idx].to(t.dtype)[spec["perm"]]
    if kind == "cmean":
        return t.mean(0, keepdim=True).expand_as(t)
    if kind == "dev":
        return t - t.mean(0, keepdim=True)
    raise ValueError(kind)


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__()
        self.w, self.eps = nn.Parameter(torch.ones(d)), eps

    def forward(self, x):
        return self.w * x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)


def rope(T, dh, device, base=10000.0):
    inv = 1.0 / (base ** (torch.arange(0, dh, 2, device=device).float() / dh))
    f = torch.outer(torch.arange(T, device=device).float(), inv)
    return f.cos(), f.sin()


def apply_rope(x, cos, sin):
    x1, x2 = x[..., ::2], x[..., 1::2]
    cos, sin = cos[None, None].to(x.dtype), sin[None, None].to(x.dtype)
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], -1).flatten(-2)


class AttentionRoute(nn.Module):
    def __init__(self, d, n_heads):
        super().__init__()
        self.h, self.dh = n_heads, d // n_heads
        self.norm = RMSNorm(d)
        self.qkv = nn.Linear(d, 3 * d, bias=False)
        self.o = nn.Linear(d, d, bias=False)
        nn.init.normal_(self.qkv.weight, std=0.02)
        nn.init.zeros_(self.o.weight)

    def forward(self, x):
        B, T, D = x.shape
        q, k, v = self.qkv(self.norm(x)).view(B, T, 3, self.h, self.dh).permute(2, 0, 3, 1, 4)
        cos, sin = rope(T, self.dh, x.device)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.o(y.transpose(1, 2).reshape(B, T, D))


class DualRouteBlock(nn.Module):
    def __init__(self, base, idx, attn=None):
        super().__init__()
        self.base, self.idx, self.attn = base, idx, attn

    def forward(self, hidden_states, *args, **kwargs):
        out = self.base(hidden_states, *args, **kwargs)
        h = out[0] if isinstance(out, tuple) else out
        ssm = apply_iv("ssm", self.idx, h - hidden_states)
        a = apply_iv("attn", self.idx, self.attn(hidden_states) if self.attn is not None else None)
        new = hidden_states + ssm + (a if a is not None else 0.0)
        return (new,) + tuple(out[1:]) if isinstance(out, tuple) else new


ATTN_LAYERS = sorted(set(np.linspace(0, N_LAYERS - 1, N_ATTN_LAYERS).round().astype(int).tolist()))
if not isinstance(LAYERS[0], DualRouteBlock):
    for i in range(N_LAYERS):
        route = AttentionRoute(D_MODEL, ATTN_HEADS).to(DEVICE) if i in ATTN_LAYERS else None
        LAYERS[i] = DualRouteBlock(LAYERS[i], i, route)

# load the Stage 1 attention weights
_ck = torch.load(PHASE_A_CKPT, map_location="cpu", weights_only=False)
_blocks = [b for b in LAYERS if b.attn is not None]
assert len(_ck["attn"]) == len(_blocks), (f"Stage 1 had {len(_ck['attn'])} attention routes, this "
                                          f"notebook built {len(_blocks)}; N_ATTN_LAYERS must match")
for blk, sd in zip(_blocks, _ck["attn"]):
    blk.attn.load_state_dict(sd)
del _ck

ATTN_PARAMS = [p for b in _blocks for p in b.attn.parameters()]
_attn_ids = {id(p) for p in ATTN_PARAMS}
BACKBONE_PARAMS = [p for p in model.parameters() if id(p) not in _attn_ids]


def set_trainable(mode):
    """'all' = whole model; 'top' = attention + the upper half of the backbone (OOM fallback);
    'attn' = attention only, which reproduces the Stage 1 frozen-backbone failure mode."""
    for p in model.parameters():
        p.requires_grad_(False)
    for p in ATTN_PARAMS:
        p.requires_grad_(True)
    if mode == "all":
        for p in model.parameters():
            p.requires_grad_(True)
    elif mode == "top":
        for i in range(N_LAYERS // 2, N_LAYERS):
            for p in LAYERS[i].parameters():
                p.requires_grad_(True)
    n = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return mode, n


UNFREEZE, N_TRAIN = set_trainable("all")
print(f"attention route restored at layers {ATTN_LAYERS} "
      f"({sum(p.numel() for p in ATTN_PARAMS) / 1e6:.1f}M) | trainable {N_TRAIN / 1e6:.1f}M "
      f"(mode '{UNFREEZE}')")

BASE_SD = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
print(f"pre-switch state held in CPU memory ({sum(v.numel() * v.element_size() for v in BASE_SD.values()) / 1e9:.2f} GB)")


@contextmanager
def intervene(**routes):
    ACTIVE.clear()
    ACTIVE.update({k: v for k, v in routes.items() if v is not None})
    try:
        yield
    finally:
        ACTIVE.clear()


def record_store():
    return {"attn": {}, "ssm": {}}


def stack_store(store):
    return {r: {i: torch.cat(v) if len(v) > 1 else v[0] for i, v in d.items()} for r, d in store.items()}


def restore_base():
    model.load_state_dict({k: v.to(DEVICE) for k, v in BASE_SD.items()})
    model.eval()

In [ ]:
# Measurement. Deletion is the headline; resampling is kept for continuity with the paper; the
# common-mode split says whether the attention route is carrying answers or an enabling bias.
def slices(n, B):
    B = max(2, B)
    sl = [(s, min(s + B, n)) for s in range(0, n, B)]
    if len(sl) > 1 and sl[-1][1] - sl[-1][0] == 1:
        sl[-2] = (sl[-2][0], n); sl.pop()
    return sl


def with_oom_retry(fn, B, label):
    while True:
        try:
            return fn(B)
        except torch.cuda.OutOfMemoryError:
            ACTIVE.clear()
            if IS_CUDA:
                torch.cuda.empty_cache()
            if B <= 2:
                raise
            B //= 2
            print(f"    out of memory in {label}; retrying with batch {B}", flush=True)


@torch.no_grad()
def last_logits(x):
    with amp():
        return model(input_ids=x, use_cache=False).logits[:, -1, :].float()


def forced_choice(lg, cand):
    return cand.gather(1, torch.gather(lg, 1, cand).argmax(-1, keepdim=True)).squeeze(1)


@torch.no_grad()
def lm_loss_eval(n_windows=8):
    W = wiki_tokens()
    k = len(W) // LM_LEN
    idx = np.random.default_rng(EVAL_SEED).integers(0, k, n_windows)
    tot, m = 0.0, 0
    for i in idx:
        x = torch.tensor(W[int(i) * LM_LEN:(int(i) + 1) * LM_LEN])[None].to(DEVICE)
        with amp():
            lg = model(input_ids=x, use_cache=False).logits[:, :-1]
        tot += float(F.cross_entropy(lg.float().reshape(-1, lg.shape[-1]), x[:, 1:].reshape(-1)))
        m += 1
    return tot / max(m, 1)


@torch.no_grad()
def measure(load, dist, B=None, n=None):
    model.eval()
    B = B or EVAL_BATCH
    X, Y, C = make_prompts(load, dist, n or N_EVAL, EVAL_SEED)
    tot = dict.fromkeys(["clean", "del_attn", "attn_res", "ssm_res", "donor", "cmean", "dev"], 0.0)
    an = sn = cm = dv = 0.0
    nb = nn_ = 0
    for i, j in slices(len(X), B):
        x, y, c = X[i:j].to(DEVICE), Y[i:j].to(DEVICE), C[i:j].to(DEVICE)
        perm = torch.roll(torch.arange(len(x), device=DEVICE), 1)
        st = record_store()
        with intervene(attn={"kind": "record", "store": st["attn"]},
                       ssm={"kind": "record", "store": st["ssm"]}):
            lg = last_logits(x)
        st = stack_store(st)
        tot["clean"] += int((forced_choice(lg, c) == y).sum())
        # write-share statistic (descriptive, the analogue of retrieval share)
        a_n = float(torch.stack([st["attn"][i2].norm(dim=-1).mean() for i2 in st["attn"]]).mean())
        s_n = float(torch.stack([st["ssm"][i2].norm(dim=-1).mean() for i2 in ATTN_LAYERS]).mean())
        an += a_n; sn += s_n
        # common-mode split at the first attention layer, as in Appendix F
        first = st["attn"][ATTN_LAYERS[0]][:, -1, :]
        cm += float(first.mean(0).norm()); dv += float((first - first.mean(0, keepdim=True)).norm(dim=-1).mean())
        nb += 1
        with intervene(attn={"kind": "zero"}):
            tot["del_attn"] += int((forced_choice(last_logits(x), c) == y).sum())
        with intervene(attn={"kind": "cache", "store": st["attn"], "perm": perm}):
            p_a = forced_choice(last_logits(x), c)
        tot["attn_res"] += int((p_a == y).sum()); tot["donor"] += int((p_a == y[perm]).sum())
        with intervene(ssm={"kind": "cache", "store": st["ssm"], "perm": perm}):
            tot["ssm_res"] += int((forced_choice(last_logits(x), c) == y).sum())
        with intervene(attn={"kind": "cmean"}):
            tot["cmean"] += int((forced_choice(last_logits(x), c) == y).sum())
        with intervene(attn={"kind": "dev"}):
            tot["dev"] += int((forced_choice(last_logits(x), c) == y).sum())
        del st
        nn_ += len(x)
        if IS_CUDA:
            torch.cuda.empty_cache()
    m = {k: v / nn_ for k, v in tot.items()}
    head = m["clean"] - 1.0 / N_CAND
    out = {"load": load, "dist": dist, "n": nn_,
           "acc": m["clean"], "acc_del_attn": m["del_attn"],
           "acc_attn_cmean": m["cmean"], "acc_attn_dev": m["dev"], "donor": m["donor"],
           "attn_write": an / nb, "ssm_write": sn / nb,
           "share_attn": an / max(an + sn, 1e-9),
           "cmean_norm": cm / nb, "dev_norm": dv / nb}
    for r, key in (("attn", "attn_res"), ("ssm", "ssm_res")):
        out[f"D_{r}"] = float("nan") if head < 0.10 else float(np.clip((m["clean"] - m[key]) / head, 0.0, 1.0))
    model.train()
    return out


def fmt(m):
    return (f"acc {m['acc']:.3f} | DELETE-attn {m['acc_del_attn']:.3f} | D_attn {m['D_attn']:.2f} "
            f"D_ssm {m['D_ssm']:.2f} | share {m['share_attn']:.3f} | cmean {m['acc_attn_cmean']:.2f} "
            f"dev {m['acc_attn_dev']:.2f}")


VIABLE = [tuple(c) for c in BOUNDARY["viable"]]
NONVIABLE = [tuple(c) for c in BOUNDARY["nonviable"]]
assert VIABLE, "the Stage 1 screen found no viable condition; there is nothing to hand back to"
# the viable condition with the largest load is the interesting one: the backbone can do it, but only just
COND_POS = max(VIABLE, key=lambda c: (c[0], c[1]))
COND_NEG = min(NONVIABLE, key=lambda c: (c[0], c[1])) if NONVIABLE else None
print(f"positive (viable) condition: {COND_POS}    negative (non-viable) condition: {COND_NEG}")

restore_base()
sync(); _t0 = time.time()
PRE = with_oom_retry(lambda B: measure(*COND_POS, B=B), EVAL_BATCH, "pre-switch measurement")
sync()
SEC_PER_EVAL = time.time() - _t0
print(f"\npre-switch state at {COND_POS}: {fmt(PRE)}")
print(f"(one evaluation takes {SEC_PER_EVAL / 60:.1f} min)")
if PRE["acc_del_attn"] > 0.8 * PRE["acc"]:
    print("\nNOTE: deleting attention barely hurts already, so the attention route never really took "
          "over. Stage 2 has nothing to hand back. Send this line back before running further.")

In [ ]:
# Price selection, BEFORE any arm runs. The penalty is the attention write magnitude, normalised by
# its pre-switch value so lambda is in comparable units across models. Writes price.json once.
PRICE_REF = max(PRE["attn_write"] ** 2, 1e-12)


def train_steps(n_steps, lam, cond, seed, price_on_until=None, trace=None, eval_every=None,
                t_deadline=None, ckpt_name=None, start=0, opt_state=None):
    """Train from the current model state. Returns (trace, step_reached)."""
    load, dist = cond
    trace = trace if trace is not None else []
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR,
                            betas=(0.9, 0.95), weight_decay=0.0)
    if opt_state is not None:
        try:
            opt.load_state_dict(opt_state)
        except Exception:
            pass
    rng = np.random.default_rng([seed, start, load])
    W = wiki_tokens()
    kw = len(W) // LM_LEN
    wiki_train = torch.tensor(W[:kw * LM_LEN]).view(kw, LM_LEN)
    model.train()
    last_ck = time.time()
    losses, pens = [], []
    for step in range(start + 1, n_steps + 1):
        for g in opt.param_groups:
            g["lr"] = LR * min(1.0, step / WARMUP)
        priced = lam > 0 and (price_on_until is None or step <= price_on_until)
        X, Y, _ = make_prompts(load, dist, TRAIN_BATCH, int(rng.integers(2 ** 31)))
        x, y = X.to(DEVICE), Y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        acc = []
        with intervene(attn={"kind": "price", "acc": acc} if priced else None):
            with amp():
                lg = model(input_ids=x, use_cache=False).logits[:, -1, :]
            loss = F.cross_entropy(lg.float(), y)
            pen = torch.stack(acc).mean() / PRICE_REF if acc else torch.zeros((), device=DEVICE)
            SCALER.scale(loss + lam * pen).backward()
        xi = wiki_train[rng.integers(0, kw, LM_BATCH)].to(DEVICE)
        with amp():
            lgl = model(input_ids=xi, use_cache=False).logits[:, :-1]
        lm = F.cross_entropy(lgl.float().reshape(-1, lgl.shape[-1]), xi[:, 1:].reshape(-1))
        SCALER.scale(LM_W * lm).backward()
        SCALER.unscale_(opt)
        nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], GRAD_CLIP)
        SCALER.step(opt)
        SCALER.update()
        losses.append(float(loss.detach())); pens.append(float(pen.detach()))
        ee = eval_every or EVAL_EVERY
        if step % ee == 0 or step == n_steps:
            m = with_oom_retry(lambda B: measure(load, dist, B=B), EVAL_BATCH, "evaluation")
            m.update({"step": step, "loss": float(np.mean(losses)), "pen": float(np.mean(pens)),
                      "priced": bool(priced), "lm": float(lm.detach())})
            trace.append(m)
            print(f"    step {step:>4} {'price on ' if priced else 'price off'} "
                  f"loss {np.mean(losses):.3f} | {fmt(m)}", flush=True)
            losses, pens = [], []
            model.train()
        if ckpt_name and time.time() - last_ck > 60 * CKPT_MINUTES and step < n_steps:
            ckpt_save(ckpt_name, {"sd": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                                  "step": step, "trace": trace})
            last_ck = time.time()
        if t_deadline and time.time() > t_deadline:
            print(f"    time budget reached at step {step}", flush=True)
            return trace, step
    return trace, n_steps


# Does full unfreezing fit on this GPU? Probe one real training step and fall back rather than
# crash mid-run. 'attn' is the last resort and is reported loudly, because it is exactly the
# frozen-backbone setting that made Stage 1 learn an enabling bias instead of a route.
def probe_step(mode):
    global UNFREEZE, N_TRAIN
    UNFREEZE, N_TRAIN = set_trainable(mode)
    restore_base()
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR)
    X, Y, _ = make_prompts(COND_POS[0], COND_POS[1], TRAIN_BATCH, SEED)
    x, y = X.to(DEVICE), Y.to(DEVICE)
    ok, t0 = True, time.time()
    try:
        model.train()
        acc = []
        with intervene(attn={"kind": "price", "acc": acc}):
            with amp():
                lg = model(input_ids=x, use_cache=False).logits[:, -1, :]
            loss = F.cross_entropy(lg.float(), y) + 0.0 * torch.stack(acc).mean()
            SCALER.scale(loss).backward()
        SCALER.unscale_(opt)
        SCALER.step(opt)
        SCALER.update()
        sync()
    except torch.cuda.OutOfMemoryError:
        ok = False
    dt = time.time() - t0
    ACTIVE.clear()
    opt.zero_grad(set_to_none=True)
    del opt
    if IS_CUDA:
        torch.cuda.empty_cache()
    return ok, dt


SEC_PER_STEP = None
for _mode in ("all", "top", "attn"):
    _ok, _dt = probe_step(_mode)
    if _ok:
        SEC_PER_STEP = _dt * 1.6      # the real step adds the text-replay forward/backward
        print(f"training mode '{_mode}': {N_TRAIN / 1e6:.1f}M trainable, about {SEC_PER_STEP:.1f} s/step")
        break
    print(f"training mode '{_mode}' does not fit in memory; falling back")
assert SEC_PER_STEP is not None, "even attention-only training runs out of memory; lower TRAIN_BATCH"
if UNFREEZE == "attn":
    print("WARNING: the backbone could not be unfrozen. Nothing can receive the computation, so a "
          "null result here repeats Stage 1 rather than testing handover. Lower TRAIN_BATCH or use "
          "a smaller backbone before believing a negative.")

# Size the run to the budget: a late/never pair should fit in roughly half of it.
for _cand in (STEER_STEPS, 300, 200, 120, 80):
    if _cand * SEC_PER_STEP + (1 + _cand / max(EVAL_EVERY, 1)) * SEC_PER_EVAL <= 0.45 * TIME_BUDGET_H * 3600:
        STEER_STEPS = _cand
        break
else:
    STEER_STEPS = 80
EVAL_EVERY = max(20, STEER_STEPS // 4)
WITHDRAW_AT = STEER_STEPS // 2
DOSE_STEPS = max(40, STEER_STEPS // 3)
print(f"steering: {STEER_STEPS} steps per arm, evaluation every {EVAL_EVERY}, withdrawal at "
      f"{WITHDRAW_AT}, dose-response {DOSE_STEPS} steps per lambda "
      f"(~{(STEER_STEPS * SEC_PER_STEP + (1 + STEER_STEPS / EVAL_EVERY) * SEC_PER_EVAL) / 60:.0f} min per arm)")
restore_base()

DOSE_SPEC = {**BASE_SPEC, "block": "dose", "cond": list(COND_POS), "steps": DOSE_STEPS,
             "lams": LAMBDAS, "lr": LR}
DOSE = REC.get(DOSE_SPEC)
if DOSE is None:
    rows = []
    print(f"\ndose-response at {COND_POS} ({DOSE_STEPS} steps per lambda):")
    for lam in LAMBDAS:
        restore_base()
        print(f"  lambda {lam}")
        tr, _ = train_steps(DOSE_STEPS, lam, COND_POS, SEED, eval_every=DOSE_STEPS)
        rows.append({"lam": lam, **{k: v for k, v in tr[-1].items()}})
    DOSE = REC.put(DOSE_SPEC, {"rows": rows})
else:
    print("dose-response finished earlier")
DR = pd.DataFrame(DOSE["rows"])
print("\n" + DR[["lam", "acc", "acc_del_attn", "share_attn", "D_attn", "D_ssm"]].round(3).to_string(index=False))

ok = DR[(DR.share_attn <= SHARE_TARGET) & (DR.acc >= ACC_FLOOR * PRE["acc"])]
LAM = float(ok.lam.min()) if len(ok) else float(DR.lam.iloc[len(DR) // 2])
PRICE = write_once(RESDIR / "price.json", {
    "created": time.strftime("%Y-%m-%d %H:%M:%S %Z"), "lambda": LAM, "ref": PRICE_REF,
    "rule": f"smallest lambda with attention write share <= {SHARE_TARGET} and accuracy >= "
            f"{ACC_FLOOR} x pre-switch ({PRE['acc']:.3f}); midpoint of the grid if none qualifies",
    "qualified": bool(len(ok)), "grid": LAMBDAS,
    "rows": [{k: float(v) for k, v in r.items() if isinstance(v, (int, float))} for r in DOSE["rows"]]})
LAM = PRICE["lambda"]
_how = "by the rule" if PRICE["qualified"] else "as the grid midpoint (NO lambda satisfied the rule - report this)"
print(f"\nlambda = {LAM} selected {_how}, recorded {PRICE['created']}")

In [ ]:
# The arms. Every job forks the same pre-switch state; `never` is the matched control that makes
# the paired comparison meaningful, so it runs immediately after each `late` job.
JOBS = []
for s in SEEDS:
    JOBS += [(COND_POS, "late", s), (COND_POS, "never", s)]
if COND_NEG is not None:
    JOBS += [(COND_NEG, "late", s) for s in SEEDS]
JOBS += [(COND_POS, "withdrawn", s) for s in SEEDS]

T_END = time.time() + TIME_BUDGET_H * 3600
RESULTS = []
print(f"\n{len(JOBS)} jobs planned; running in priority order until the {TIME_BUDGET_H} h budget is spent\n")
for cond, arm, seed in JOBS:
    spec = {**BASE_SPEC, "block": "arm", "cond": list(cond), "arm": arm, "seed": seed,
            "lam": LAM if arm != "never" else 0.0, "steps": STEER_STEPS,
            "withdraw_at": WITHDRAW_AT if arm == "withdrawn" else None, "lr": LR}
    got = REC.get(spec)
    if got is not None:
        RESULTS.append(got); print(f"[{arm} {cond} seed {seed}] finished earlier"); continue
    if time.time() > T_END:
        print(f"[{arm} {cond} seed {seed}] skipped: time budget spent"); continue
    print(f"[{arm} {cond} seed {seed}] lambda {spec['lam']}")
    name = f"arm_{keyof(spec)}"
    ck = ckpt_load(name)
    restore_base()
    start, trace = 0, []
    if ck is not None:
        model.load_state_dict({k: v.to(DEVICE) for k, v in ck["sd"].items()})
        start, trace = ck["step"], ck["trace"]
        print(f"    resumed at step {start}/{STEER_STEPS}")
    else:
        pre = with_oom_retry(lambda B: measure(*cond, B=B), EVAL_BATCH, "pre")
        pre.update({"step": 0, "loss": float("nan"), "pen": float("nan"), "priced": False,
                    "lm": float("nan")})
        trace.append(pre)
        print(f"    step    0 pre-switch  {fmt(pre)}", flush=True)
    trace, reached = train_steps(
        STEER_STEPS, spec["lam"], cond, seed, price_on_until=spec["withdraw_at"],
        trace=trace, t_deadline=T_END, ckpt_name=name, start=start)
    if reached >= STEER_STEPS:
        lm = lm_loss_eval()
        with intervene(attn={"kind": "zero"}):
            lm_del = lm_loss_eval()
        RESULTS.append(REC.put(spec, {"trace": trace, "lm_loss": lm, "lm_loss_del_attn": lm_del}))
        (CKPTDIR / f"{name}.pt").unlink(missing_ok=True)
        print(f"    done. wikitext loss {lm:.3f} clean, {lm_del:.3f} with attention deleted\n", flush=True)
    else:
        ckpt_save(name, {"sd": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                         "step": reached, "trace": trace})
        print("    paused; run all again to continue\n", flush=True)
        break

In [ ]:
# Summary: the paired late-vs-never comparison, the boundary row, and the verdict.
rows = []
for r in RESULTS:
    s, t = r["_spec"], r["trace"]
    a, b = t[0], t[-1]
    rows.append({"cond": f"L{s['cond'][0]}_d{s['cond'][1]}", "arm": s["arm"], "seed": s["seed"], "lam": s["lam"],
                 "acc0": a["acc"], "acc": b["acc"], "del0": a["acc_del_attn"], "del": b["acc_del_attn"],
                 "share0": a["share_attn"], "share": b["share_attn"],
                 "D_attn": b["D_attn"], "D_ssm": b["D_ssm"],
                 "cmean": b["acc_attn_cmean"], "dev": b["acc_attn_dev"],
                 "lm": r.get("lm_loss", float("nan")), "lm_del": r.get("lm_loss_del_attn", float("nan"))})
RS = pd.DataFrame(rows)

lines = [f"=== STAGE 2: HANDOVER BACK TO A PRETRAINED RECURRENT BACKBONE | {time.strftime('%Y-%m-%d %H:%M')} ===",
         f"model {MODEL_NAME} | {GPU_NAME} | attention at layers {ATTN_LAYERS} | backbone UNFROZEN during steering",
         f"boundary recorded {BOUNDARY['created']}: viable {BOUNDARY['viable']}, not viable {BOUNDARY['nonviable']}",
         f"positive condition {COND_POS} | negative condition {COND_NEG}",
         f"price recorded {PRICE['created']}: lambda {LAM} (rule satisfied: {PRICE['qualified']})",
         f"pre-switch: {fmt(PRE)}",
         "",
         "-- dose-response (selection, run before the arms) --",
         DR[["lam", "acc", "acc_del_attn", "share_attn", "D_attn", "D_ssm"]].round(3).to_string(index=False),
         ""]

if len(RS):
    lines += ["-- arms (acc0/del0 = pre-switch, acc/del = endpoint; DELETION is the headline) --",
              RS.round(3).to_string(index=False), ""]
    RS["handover"] = ((RS["del"] >= HANDOVER_DEL * RS["acc"]) & (RS["acc"] >= HANDOVER_ACC * RS["acc0"]))
    POS_TAG = f"L{COND_POS[0]}_d{COND_POS[1]}"
    for cond in RS["cond"].unique():
        sub = RS[RS["cond"] == cond]
        lab = "VIABLE" if cond == POS_TAG else "NON-VIABLE"
        for arm in sub["arm"].unique():
            q = sub[sub["arm"] == arm]
            lines.append(f"  {lab} {cond} {arm:<10} handover {int(q['handover'].sum())}/{len(q)} | "
                         f"deletion acc {q['del'].mean():.3f} +- {q['del'].std(ddof=0):.3f} | "
                         f"clean acc {q['acc'].mean():.3f} | D_ssm {q['D_ssm'].mean():.2f} "
                         f"D_attn {q['D_attn'].mean():.2f}")
    pair = RS[(RS["cond"] == POS_TAG) & RS["arm"].isin(["late", "never"])]
    piv = pair.pivot_table(index="seed", columns="arm", values="del")
    if {"late", "never"} <= set(piv.columns):
        d = (piv["late"] - piv["never"]).dropna()
        lines += ["", f"paired late - never on deletion accuracy, {len(d)} seeds: "
                      f"{d.mean():+.3f} +- {d.std(ddof=0):.3f}  (per seed: {[round(v, 3) for v in d]})",
                  "  this is the number that answers 'maybe the backbone just relearned it'"]
    lines += ["", "verdict: " + (
        "PRETRAINED HANDOVER" if RS[(RS["cond"] == POS_TAG) & (RS["arm"] == "late")]["handover"].any()
        else "NO HANDOVER -- report as a boundary row: the screen passed and steering still failed")]
else:
    lines += ["no arm finished in the time budget; run all again to continue"]

mpl.rcParams.update({"figure.dpi": 130, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 8,
                     "axes.spines.top": False, "axes.spines.right": False, "legend.fontsize": 7})
if len(RESULTS):
    fig, axes = plt.subplots(1, 3, figsize=(7.6, 2.4))
    for r in RESULTS:
        s, T = r["_spec"], pd.DataFrame(r["trace"])
        lab = f"{s['arm']} {tuple(s['cond'])} s{s['seed']}"
        axes[0].plot(T.step, T.acc, "-o", ms=2.5, label=lab)
        axes[1].plot(T.step, T.acc_del_attn, "-o", ms=2.5, label=lab)
        axes[2].plot(T.step, T.D_ssm, "-o", ms=2.5, label=lab)
        axes[2].plot(T.step, T.D_attn, ":s", ms=2.5)
    for ax, ttl in zip(axes, ("clean accuracy", "accuracy with attention DELETED",
                              "causal dependence (solid SSM, dotted attn)")):
        ax.set_xlabel("steps after switch"); ax.set_title(ttl); ax.set_ylim(-0.03, 1.03)
    axes[0].axhline(1 / N_CAND, color="0.7", lw=0.6, ls=":")
    h, l = axes[1].get_legend_handles_labels()
    fig.tight_layout()
    fig.legend(h, l, loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.0))
    for ext in ("pdf", "png"):
        fig.savefig(FIGDIR / f"stage2.{ext}")
    plt.show()

SUMMARY = "\n".join(lines)
(TABDIR / "STAGE2_SUMMARY.txt").write_text(SUMMARY)
if len(RS):
    RS.to_csv(TABDIR / "stage2_arms.csv", index=False)
DR.to_csv(TABDIR / "stage2_dose.csv", index=False)
print(SUMMARY)
print(f"\nsaved {TABDIR / 'STAGE2_SUMMARY.txt'}")